# SuperRaman classification contract

## Goal
Inspect logits and explicit input scaling; select research checkpoints on separate Renishaw validation.

## Setup
Install the package in this kernel's environment. All default data are synthetic; research training is disabled.

In [ ]:
from pathlib import Path
import os, sys, uuid
import numpy as np
PROJECT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").exists()), Path.cwd())
WORK = Path(os.environ.get("SERS_NOTEBOOK_OUTPUT", PROJECT / "output/notebooks")) / uuid.uuid4().hex[:8]
WORK.mkdir(parents=True)
from sersd2d_superraman.data import SpectrumBatch, load_batch, save_batch
from sersd2d_superraman.cli import synthetic, prepare

### 1. Inspect the compatible model

In [ ]:
import torch
from sersd2d_superraman.classification.model import SuperRaman, predict_logits
from sersd2d_superraman.preprocessing import classifier_input
torch.manual_seed(37)
model = SuperRaman().eval()
synthetic(WORK / "synthetic", per_class=5)
batch = load_batch(WORK / "synthetic/Renishaw.npz")
logits = predict_logits(model, batch.spectra[:3])
{"parameters": sum(x.numel() for x in model.parameters()), "input_shape": classifier_input(batch.spectra[:3]).shape, "logit_shape": logits.shape}

## Checks

In [ ]:
assert logits.shape == (3,20) and np.isfinite(logits).all()
assert sum(x.numel() for x in model.parameters()) == 448572

### 2. Research training command

In [ ]:
RUN_RESEARCH_TRAINING = False
command = [sys.executable, "-m", "sersd2d_superraman", "train-classifier", "--data", "data/prepared", "--config", "configs/maintained/classifier.json", "--output", "output/classifier"]
if RUN_RESEARCH_TRAINING:
    import subprocess
    subprocess.run(command, cwd=PROJECT, check=True)
else:
    print("Prepared command:", " ".join(command))

## Next steps
This initialized model has no established accuracy. Use the documented split preparation, then evaluate held-out data once selection finishes. Choose a separate shift profile explicitly when changing legacy behavior.